In [ ]:
# ---------------- 1. PAKET KURULUMLARI ----------------
!pip install -q transformers accelerate datasets scikit-learn matplotlib seaborn

In [ ]:
# ---------------- 2. KÜTÜPHANELER ----------------
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [ ]:
# ---------------- 3. HUGGING FACE GİRİŞİ ----------------
from google.colab import userdata
hf_token = userdata.get("hf")
login(token=hf_token)

In [ ]:
# ---------------- 4. VERİ SETİNİ YÜKLEME, TEMİZLEME VE AYIRMA ----------------
print("\nVeri seti Hugging Face Hub üzerinden indiriliyor...")

dataset_path = hf_hub_download(
    repo_id="nihalenc/turkish-8class-emotion-dataset",
    filename="turkish-8class-emotion-dataset.csv",
    repo_type="dataset"
)

df = pd.read_csv(dataset_path, sep=";", encoding="utf-8")
df.columns = df.columns.str.strip()

def metin_temizle(text):
    if not isinstance(text, str):
        return ""

    # URL veya link içeren parçalar silinir.
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    # Kullanıcı mentionları silinir.
    text = re.sub(r'\@\w+', '', text)
    # HTML etiketlerini siler
    text = re.sub(r'<.*?>', '', text)
    # Hashtag tamamen silinmez; duygu bilgisi taşıyabileceği için etiket_ biçimine çevrilir.
    text = re.sub(r'#(\w+)', r'etiket_\1', text)

    # Üç veya daha fazla tekrar eden harf iki harfe indirilir.
    # Örn: "çoook" → "çook"
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return text.strip()

df['text'] = df['text'].apply(metin_temizle)

print("Duplicate silmeden önce:", len(df))
df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print("Duplicate sildikten sonra:", len(df))

emotions = [
    'igrenme',
    'korku',
    'minnet',
    'pismanlik',
    'mutluluk',
    'ofke',
    'saskinlik',
    'uzuntu'
]

label2id = {label: i for i, label in enumerate(emotions)}
id2label = {i: label for i, label in enumerate(emotions)}
df['label'] = df[emotions].idxmax(axis=1).map(label2id)

# Verinin %80'i eğitim, %20'si geçici set olarak ayrılır.
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# Geçici set ikiye bölünerek validation ve test setleri oluşturulur.
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

# --- EĞİTİM VERİSİNİN HAZIRLANMASI ---
train_df_balanced = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


print(f"Eğitim seti boyutu: {len(train_df_balanced)}")

train_dataset = Dataset.from_pandas(train_df_balanced)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

def dagilimi_goster(df_girdi, set_adi):
   
    sayimlar = df_girdi['label'].map(id2label).value_counts()

    print(f" --- {set_adi} Seti ---")
    print(sayimlar.to_string())
    print(f"Toplam: {len(df_girdi)}\n")


dagilimi_goster(train_df_balanced, "EĞİTİM (Train)")
dagilimi_goster(val_df, "DOĞRULAMA (Validation)")
dagilimi_goster(test_df, "TEST (Test)")

print(" --- TOPLAM VERİ SETİ ---")

# Bölünmeden önceki tüm veri setindeki genel duygu dağılımı hesaplanır.
toplam_sayimlar = df['label'].map(id2label).value_counts()
print(toplam_sayimlar.to_string())
print(f"Genel Toplam: {len(df)}\n")

In [ ]:
# ---------------- 5. MODEL KONFİGÜRASYONLARI ----------------
MODEL_CONFIGS = {
    'bert': {
        'model_name'                 : 'dbmdz/bert-base-turkish-cased',
        'max_length'                 : 128,
        'learning_rate'              : 2e-5,
        'weight_decay'               : 0.05,
        'warmup_ratio'               : 0.1,
        'warmup_steps'               : None,
        'batch_size'                 : 32,
        'epochs'                     : 5,
        'gradient_accumulation_steps': 1,
        'alpha_weights'              : [1.0, 1.0, 1.0, 1.0, 1.0, 1.2, 1.2, 1.5],
        'patience'                   : 3,
        'lr_scheduler_type'          : 'cosine',
        'output_dir'                 : '/content/berturk_final',
        'hub_model_id'               : 'nihalenc/berturk-duygu-analizi',
    },

    'electra': {
        'model_name'                 : 'dbmdz/electra-base-turkish-cased-discriminator',
        'max_length'                 : 128,
        'learning_rate'              : 2e-5,
        'weight_decay'               : 0.05,
        'warmup_ratio'               : 0.1,
        'warmup_steps'               : None,
        'batch_size'                 : 16,
        'epochs'                     : 5,
        'gradient_accumulation_steps': 4,
        'alpha_weights'              : [1.0, 1.0, 1.0, 1.0, 0.9, 1.2, 1.2, 1.5],
        'patience'                   : 3,
        'lr_scheduler_type'          : 'cosine',
        'output_dir'                 : '/content/electra_final',
        'hub_model_id'               : 'nihalenc/electra-duygu-analizi',
    },
}


In [ ]:
# ---------------- 6. METRİK HESAPLAMA VE ÖZEL TRAINER SINIFI ----------------
def compute_metrics(eval_pred):
    # Model her sınıf için logit üretir; en yüksek logit tahmin edilen sınıf seçilir.
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids

    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy'        : accuracy_score(labels, preds),
        'f1_macro'        : f1,
        'precision_macro' : p,
        'recall_macro'    : r,
    }

class EmotionTrainer(Trainer):

    def __init__(self, alpha_tensor=None, uzuntu_idx=None, ofke_idx=None, *args, **kwargs):
       
        self.alpha_tensor = alpha_tensor if alpha_tensor is not None else torch.tensor([1.0, 1.0, 1.0, 1.0, 0.9, 1.0, 1.2, 1.5], dtype=torch.float32).to("cuda")
        self.uzuntu_idx = uzuntu_idx
        self.ofke_idx = ofke_idx

        unexpected_keys = ['processing_class']
        for key in unexpected_keys:
            if key in kwargs:
                kwargs.pop(key)

        super().__init__(*args, **kwargs)

    # Focal loss kolay örneklerin etkisini azaltıp zor örneklere daha fazla ağırlık verir.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

       
        ce_loss = F.cross_entropy(logits.view(-1, self.model.config.num_labels),
                                  labels.view(-1),
                                  reduction='none')

        
        alpha_tensor = self.alpha_tensor.to(logits.device)
        batch_alpha = alpha_tensor[labels.view(-1)]
        pt = torch.exp(-ce_loss)
        focal_loss = (batch_alpha * ((1 - pt) ** 2.0) * ce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss


In [ ]:
# ---------------- 7. MODEL EĞİTİM FONKSİYONU ----------------
def train_emotion_model(model_type, train_dataset, val_dataset, test_dataset,
                        emotions, id2label, label2id):
    cfg = MODEL_CONFIGS[model_type]
    print(f"\n{'═' * 62}")
    print(f"  {cfg['model_name']}")
    print(f"  LR={cfg['learning_rate']} | WD={cfg['weight_decay']} "
          f"| max_len={cfg['max_length']} | epochs={cfg['epochs']}")
    print(f"{'═' * 62}\n")

    tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
    # Önceden eğitilmiş modelin üzerine 8 sınıflı sınıflandırma katmanı eklenir.
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg['model_name'],
        num_labels=len(emotions),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    def preprocess(examples):
        return tokenizer(examples["text"], truncation=True,
                         padding="max_length", max_length=cfg['max_length'])

    # Train, validation ve test setleri aynı tokenizer ayarlarıyla sayısal forma çevrilir.
    tok_train = train_dataset.map(preprocess, batched=True)
    tok_val   = val_dataset.map(preprocess,   batched=True)
    tok_test  = test_dataset.map(preprocess,  batched=True)

    alpha_tensor = torch.tensor(cfg['alpha_weights'], dtype=torch.float32).to("cuda")
    grad_accum  = cfg.get('gradient_accumulation_steps', 1)
    eff_batch   = cfg['batch_size'] * grad_accum
    total_steps = (len(train_dataset) // eff_batch) * cfg['epochs']
    warmup_steps = (int(total_steps * cfg['warmup_ratio'])
                    if cfg.get('warmup_ratio') else cfg.get('warmup_steps', 0))
    print(f"  warmup_steps = {warmup_steps}  |  total_steps ≈ {total_steps}")

    # Hugging Face Trainer'ın eğitim, kayıt, değerlendirme ve Hub ayarları burada tanımlanır.
    training_args = TrainingArguments(
        output_dir=cfg["output_dir"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        warmup_ratio=cfg["warmup_ratio"],
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=250,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=100,
        fp16=True,
        bf16=False,
        report_to="none",
        seed=42,
        data_seed=42,
        push_to_hub=True,
        hub_model_id=cfg["hub_model_id"],
        hub_private_repo=True,
        hub_token=hf_token
    )

    trainer = EmotionTrainer(
        alpha_tensor=alpha_tensor,
        uzuntu_idx=label2id['uzuntu'],
        ofke_idx=label2id['ofke'],
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['patience'])],
    )
    trainer.train()

    trainer.push_to_hub(commit_message=f"Final {model_type} duygu analizi modeli")
    tokenizer.push_to_hub(cfg["hub_model_id"])

    return trainer, tokenizer, tok_test


In [ ]:
# ---------------- 8. TEST DEĞERLENDİRME VE RAPORLAMA FONKSİYONU ----------------
# Test setinde metrikleri hesaplar, confusion matrix çizer ve sınıflandırma raporunu kaydeder.
def evaluate_and_plot(trainer, tok_test, emotions, title="Model", cmap="Blues"):

    test_results = trainer.evaluate(tok_test)
    print(f"\n{'─' * 50}")
    print(f"  {title} — FİNAL TEST SONUÇLARI")
    print(f"{'─' * 50}")
    for k, v in test_results.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    preds_out = trainer.predict(tok_test)
    y_pred = np.argmax(preds_out.predictions, axis=1)
    y_true = preds_out.label_ids

    # Confusion matrix modelin hangi sınıfları birbirine karıştırdığını gösterir.
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d',
                cmap=cmap, xticklabels=emotions, yticklabels=emotions)
    plt.xlabel('Tahmin Edilen')
    plt.ylabel('Gerçek Değer')
    plt.title(f'{title} — Hata Matrisi (Test Seti)')
    plt.tight_layout()
    plt.show()

    # Sınıf bazlı precision, recall ve F1 değerleri raporlanır.
    report_dict = classification_report(
    y_true,
    y_pred,
    target_names=emotions,
    output_dict=True,
    zero_division=0
)

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(f"/content/{title}_classification_report.csv")
    display(report_df)
    print(classification_report(y_true, y_pred, target_names=emotions))
    return y_true, y_pred


In [ ]:
# ---------------- 9. BERTURK MODELİNİ EĞİTME ----------------
bert_trainer, bert_tokenizer, bert_tok_test = train_emotion_model(
    model_type='bert',
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    emotions=emotions,
    id2label=id2label,
    label2id=label2id,
)

trainer        = bert_trainer
tokenizer      = bert_tokenizer
tokenized_test = bert_tok_test
model          = trainer.model

In [ ]:
# ---------------- 10. BERTURK TEST DEĞERLENDİRMESİ ----------------
# BERTurk — Test Seti Değerlendirme + Confusion Matrix
# BERTurk için test sonuçları, confusion matrix ve classification report üretilir.
bert_y_true, bert_y_pred = evaluate_and_plot(
    trainer, tokenized_test, emotions, title="BERTurk", cmap="Blues"
)

In [ ]:
# ---------------- 11. BERTURK ÜZÜNTÜ-ÖFKE HATA ANALİZİ ----------------
# Üzüntü ve öfke sınıfları arasında yapılan özel karışıklıkları incelemek için analiz fonksiyonu.
def analyze_uzuntu_ofke_errors(y_true, y_pred, test_df, emotions, model_name="Model", sample_n=30):
    # İlgili sınıfların listedeki sayısal ID'leri bulunur.
    uzuntu_idx = emotions.index("uzuntu")
    ofke_idx = emotions.index("ofke")

    analysis_df = pd.DataFrame({
        "text": test_df["text"].tolist(),
        "true_id": y_true,
        "pred_id": y_pred
    })

    analysis_df["true_label"] = analysis_df["true_id"].map(lambda x: emotions[x])
    analysis_df["pred_label"] = analysis_df["pred_id"].map(lambda x: emotions[x])

    pair_errors = analysis_df[
        ((analysis_df["true_label"] == "uzuntu") & (analysis_df["pred_label"] == "ofke")) |
        ((analysis_df["true_label"] == "ofke") & (analysis_df["pred_label"] == "uzuntu"))
    ].copy()

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(emotions))))

    uzuntu_total = cm[uzuntu_idx].sum()
    ofke_total = cm[ofke_idx].sum()

    uzuntu_to_ofke_count = cm[uzuntu_idx, ofke_idx]
    ofke_to_uzuntu_count = cm[ofke_idx, uzuntu_idx]

    uzuntu_to_ofke_rate = uzuntu_to_ofke_count / uzuntu_total
    ofke_to_uzuntu_rate = ofke_to_uzuntu_count / ofke_total

    print(f"\n{'='*60}")
    print(f"{model_name} — Üzüntü / Öfke Karışıklık Analizi")
    print(f"{'='*60}")

    print(f"Gerçek üzüntü olup öfke tahmin edilen örnek sayısı: {uzuntu_to_ofke_count}")
    print(f"Gerçek üzüntü içinde öfke tahmin oranı: %{uzuntu_to_ofke_rate*100:.2f}")

    print(f"\nGerçek öfke olup üzüntü tahmin edilen örnek sayısı: {ofke_to_uzuntu_count}")
    print(f"Gerçek öfke içinde üzüntü tahmin oranı: %{ofke_to_uzuntu_rate*100:.2f}")

    print(f"\nToplam öfke-üzüntü karışıklığı: {len(pair_errors)} örnek")

    pair_errors["manual_review"] = ""
    pair_errors["review_note"] = ""

    display_cols = ["text", "true_label", "pred_label", "manual_review", "review_note"]

    if len(pair_errors) > 0:
        display(pair_errors[display_cols].sample(
            min(sample_n, len(pair_errors)),
            random_state=42
        ))
    else:
        print("Üzüntü-öfke karışıklığı bulunamadı.")

    return pair_errors

# BERTurk tahminleri üzerinde üzüntü-öfke hata analizi çalıştırılır.
bert_pair_errors = analyze_uzuntu_ofke_errors(
    y_true=bert_y_true,
    y_pred=bert_y_pred,
    test_df=test_df,
    emotions=emotions,
    model_name="BERTurk",
    sample_n=30
)

In [ ]:
# ---------------- 12. BERTURK SINIF BAZLI F1 ANALİZİ ----------------
f1_per_class = f1_score(bert_y_true, bert_y_pred, average=None)
# F1 değerleri eşiklere göre renklendirilir; düşük sınıflar görsel olarak öne çıkar.
colors = ['#e74c3c' if f < 0.70 else '#f39c12' if f < 0.80 else '#2ecc71'
          for f in f1_per_class]

# Aynı figürde sınıf bazlı F1 grafiği ve üzüntü-öfke karışıklık grafiği gösterilir.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(emotions, f1_per_class, color=colors, edgecolor='black', linewidth=0.5)
axes[0].axhline(y=0.70, color='red',  linestyle='--', linewidth=1.5, label='Kritik Eşik (0.70)')
axes[0].axhline(y=f1_per_class.mean(), color='blue', linestyle=':', linewidth=1.5,
                label=f'Macro Ort. ({f1_per_class.mean():.3f})')
axes[0].set_ylim(0.4, 1.0)
axes[0].set_ylabel('F1-Score')
axes[0].set_title('BERTurk — Sınıf Bazlı F1')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

uzuntu_idx = emotions.index('uzuntu')
ofke_idx   = emotions.index('ofke')
# Sadece gerçek etiketi üzüntü veya öfke olan örnekler seçilir.
pair_true  = [t for t, p in zip(bert_y_true, bert_y_pred) if t in (uzuntu_idx, ofke_idx)]
pair_pred  = [p for t, p in zip(bert_y_true, bert_y_pred) if t in (uzuntu_idx, ofke_idx)]
# normalize='true' her gerçek sınıf satırını yüzde/oran olarak okumayı sağlar.
pair_cm = confusion_matrix(pair_true, pair_pred, labels=[uzuntu_idx, ofke_idx], normalize='true')
sns.heatmap(pair_cm, annot=True, fmt='.2%', cmap='Reds',
            xticklabels=['uzuntu', 'ofke'], yticklabels=['uzuntu', 'ofke'],
            ax=axes[1])
axes[1].set_title('Üzüntü ↔ Öfke Karışıklık (normalize)')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gerçek')

plt.tight_layout()
plt.show()

print("\nSınıf bazlı F1:")
# Her sınıfın F1 değeri yazdırılır; düşük olanlara uyarı eklenir.
for emo, f1 in zip(emotions, f1_per_class):
    flag = "  DÜŞÜK" if f1 < 0.70 else ""
    print(f"  {emo:<12}: {f1:.4f}{flag}")

In [ ]:
# ---------------- 13. ELECTRA MODELİNİ EĞİTME ----------------
# ELECTRA modeli seçilerek aynı eğitim fonksiyonu ELECTRA ayarlarıyla çalıştırılır.
electra_trainer, electra_tokenizer, electra_tok_test = train_emotion_model(
    model_type='electra',
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    emotions=emotions,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# ---------------- 14. ELECTRA TEST DEĞERLENDİRMESİ ----------------
# ELECTRA — Test Seti Değerlendirme + Confusion Matrix + Sınıf Analizi
# ELECTRA için test sonuçları, confusion matrix ve classification report üretilir.
electra_y_true, electra_y_pred = evaluate_and_plot(
    electra_trainer, electra_tok_test, emotions, title="ELECTRA", cmap="Oranges"
)

In [ ]:
# ---------------- 15. BERTURK VE ELECTRA SINIF BAZLI F1 KARŞILAŞTIRMASI ----------------
# Sınıf bazlı F1 karşılaştırması: BERTurk vs ELECTRA

# İki modelin sınıf bazlı F1 değerleri karşılaştırma için hesaplanır.
f1_bert    = f1_score(bert_y_true,    bert_y_pred,    average=None)
f1_electra = f1_score(electra_y_true, electra_y_pred, average=None)

# Bar grafikte her duygu sınıfı için BERTurk ve ELECTRA yan yana gösterilir.
x = range(len(emotions))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - width/2 for i in x], f1_bert,    width, label='BERTurk',  color='steelblue',  alpha=0.85)
ax.bar([i + width/2 for i in x], f1_electra, width, label='ELECTRA',  color='darkorange', alpha=0.85)
ax.axhline(y=0.70, color='red', linestyle='--', linewidth=1.2, label='Kritik Eşik (0.70)')
ax.set_xticks(list(x))
ax.set_xticklabels(emotions, rotation=30)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('F1-Score')
ax.set_title('BERTurk vs ELECTRA — Sınıf Bazlı F1 Karşılaştırması')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n{'Sınıf':<12} {'BERTurk':>10} {'ELECTRA':>10} {'Delta':>8}")
print("─" * 44)
# Delta değeri ELECTRA'nın BERTurk'e göre sınıf bazında artış/azalışını gösterir.
for emo, fb, fe in zip(emotions, f1_bert, f1_electra):
    delta = fe - fb
    sign  = "▲" if delta > 0 else "▼" if delta < 0 else "─"
    print(f"  {emo:<10} {fb:>10.4f} {fe:>10.4f}   {sign}{abs(delta):.4f}")

In [ ]:
# ---------------- 16. BERTURK VE ELECTRA GENEL PERFORMANS KARŞILAŞTIRMASI ----------------
# Test setinde BERTurk ve ELECTRA'nın genel metrikleri ayrı ayrı hesaplanır.
bert_results = bert_trainer.evaluate(bert_tok_test)
electra_results = electra_trainer.evaluate(electra_tok_test)

# Sonuçlar tablo haline getirilerek iki modelin genel performansı karşılaştırılır.
comparison_df = pd.DataFrame([
    {
        "Model": "BERTurk",
        "Accuracy": bert_results["eval_accuracy"],
        "Macro F1": bert_results["eval_f1_macro"],
        "Macro Precision": bert_results["eval_precision_macro"],
        "Macro Recall": bert_results["eval_recall_macro"],
    },
    {
        "Model": "ELECTRA",
        "Accuracy": electra_results["eval_accuracy"],
        "Macro F1": electra_results["eval_f1_macro"],
        "Macro Precision": electra_results["eval_precision_macro"],
        "Macro Recall": electra_results["eval_recall_macro"],
    }
])

display(comparison_df)

In [ ]:
# ---------------- 17. INFERENCE TIME ÖLÇME ----------------
# Modelin tek cümle için ortalama tahmin süresini ölçer.
def measure_inference_time(trainer, tokenizer, texts, max_length=128, n=100):
    # Trainer içindeki eğitilmiş model alınır ve değerlendirme moduna geçirilir.
    model = trainer.model
    model.eval()

    device = next(model.parameters()).device
    # Ölçüm için test metinlerinden ilk n örnek kullanılır.
    sample_texts = texts[:n]

    times = []

    for text in sample_texts:
        # Her cümle için başlangıç zamanı alınır.
        start = time.time()

        # Metin tokenize edilip PyTorch tensor formatına çevrilir.
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        # Tahmin sırasında gradyan hesaplanmaz; bu işlem hız ve bellek açısından daha verimlidir.
        with torch.no_grad():
            _ = model(**inputs)

        end = time.time()
        # Cümlenin toplam tahmin süresi listeye eklenir.
        times.append(end - start)

    # Tüm örneklerin ortalama tahmin süresi döndürülür.
    return np.mean(times)

# BERTurk için ortalama inference süresi ölçülür.
bert_time = measure_inference_time(
    bert_trainer,
    bert_tokenizer,
    test_df["text"].tolist(),
    max_length=128,
    n=100
)

# ELECTRA için ortalama inference süresi ölçülür.
electra_time = measure_inference_time(
    electra_trainer,
    electra_tokenizer,
    test_df["text"].tolist(),
    max_length=128,
    n=100
)

print(f"BERTurk ortalama tahmin süresi: {bert_time:.4f} sn/cümle")
print(f"ELECTRA ortalama tahmin süresi: {electra_time:.4f} sn/cümle")